<div style="text-align: center; margin-bottom: 20px;">
    <img src="https://upload.wikimedia.org/wikipedia/commons/b/bd/Indian_Space_Research_Organisation_Logo.svg" width="150"/>
    <h1 style="color: #0c1844;">Lumina: ISRO Lunar Digital Twin AI Pipeline</h1>
    <h3 style="color: #ff6a00;">Smart India Hackathon (SIH26166)</h3>
    <h4 style="color: #666;">Lumina represented by LumaInit</h4>\n</div>

### **Multi-Modal, Sun-Angle, and Scale-Invariant Image Correspondence**

This notebook represents the AI backend for **Lumina**, a real-time 3D digital twin for ISRO lunar missions. It demonstrates our approach to the SIH26166 problem statement: co-registering vastly different optical products from Chandrayaan-2 (OHRC, TMC, and IIRS).

**Our Architecture encompasses:**
1. **Scale-Invariant Feature Matching:** Using LoFTR (Local Feature Transformer) to match features across massive scale differences (e.g., OHRC at 0.25m/px vs TMC at 5m/px).
2. **Subsurface Ice Classification:** Machine learning models predicting ice probability based on DFSAR (L-band/S-band) and thermal data.
3. **Space Data Integration:** Live telemetry simulation for the Shiv Shakti Point landing site.

> **Reference Repository:** [saichintamani/Lumina-](https://github.com/saichintamani/Lumina-)

---
## 🛠️ 1. Environment Setup & Dependencies
We use PyTorch and Kornia for deep learning, and Scikit-Learn for random forest classification.

In [ ]:
!pip install -q kornia torch torchvision opencv-python-headless matplotlib pandas scikit-learn tqdm

In [ ]:
import numpy as np
import pandas as pd
import cv2
import torch
import kornia
from kornia.feature import LoFTR
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import seaborn as sns
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Check hardware acceleration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🚀 Computation Device: {device.type.upper()}')
print(f'📦 PyTorch Version: {torch.__version__} | Kornia: {kornia.__version__}')

---
## 🌖 2. Multi-Modal Image Simulation (TMC vs OHRC)

The core challenge is matching **TMC (Terrain Mapping Camera)** images with **OHRC (Orbiter High Resolution Camera)** images. 
- TMC provides wide-swath, lower-resolution context (~5m/pixel).
- OHRC provides narrow-swath, ultra-high-resolution detail (~0.25m/pixel).

Here, we generate a synthetic lunar landscape and extract two views mimicking this scale and illumination variance.

In [ ]:
def create_lunar_terrain(size=1024):
    """Generates a synthetic lunar terrain with craters, regolith noise, and ridges."""
    np.random.seed(42)
    # Base regolith (noise)
    terrain = np.random.normal(120, 25, (size, size))
    terrain = cv2.GaussianBlur(terrain, (15, 15), 5)
    
    # Add Impact Craters
    y, x = np.ogrid[-size//2:size//2, -size//2:size//2]
    for _ in range(40):
        cx, cy = np.random.randint(0, size, 2)
        r = np.random.randint(15, 100)
        mask = ((x + size//2 - cx)**2 + (y + size//2 - cy)**2) < r**2
        terrain[mask] -= np.random.randint(10, 50)  # Shadow inside crater
        
        # Crater rim highlight
        rim_mask = (((x + size//2 - cx)**2 + (y + size//2 - cy)**2) >= r**2) & \
                   (((x + size//2 - cx)**2 + (y + size//2 - cy)**2) < (r+5)**2)
        terrain[rim_mask] += np.random.randint(20, 60)
        
    return np.clip(terrain, 0, 255).astype(np.float32)

# 1. Create Base Terrain
base_terrain = create_lunar_terrain(1024)

# 2. Simulate TMC (Wide field, lower resolution, scaled down)
tmc_img = cv2.resize(base_terrain, (256, 256), interpolation=cv2.INTER_AREA)

# 3. Simulate OHRC (Narrow field, high resolution, cropped from center)
center = 1024 // 2
crop_size = 150 # Narrow FOV
ohrc_raw = base_terrain[center-crop_size:center+crop_size, center-crop_size:center+crop_size]

# Add sun-angle variance to OHRC (simulating different pass time)
ohrc_img = cv2.convertScaleAbs(ohrc_raw, alpha=1.2, beta=20)
ohrc_img = cv2.resize(ohrc_img, (256, 256), interpolation=cv2.INTER_CUBIC)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(tmc_img, cmap='gray')
axes[0].set_title('Simulated TMC (Wide Swath, 5m/px)', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(ohrc_img, cmap='gray')
axes[1].set_title('Simulated OHRC (Narrow Swath, 0.25m/px)', fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

---
## 🤖 3. Deep Learning Feature Matching (LoFTR)

Traditional feature detectors (like SIFT or ORB) struggle heavily with the featureless, repetitive nature of lunar regolith and extreme scale variances.

We deploy **LoFTR (Local Feature Transformer)**. Being detector-free, it extracts dense feature maps and uses self and cross-attention mechanisms to find correspondences even in low-texture areas.

In [ ]:
# Initialize LoFTR
matcher = LoFTR(pretrained='outdoor').eval().to(device)

def extract_and_match(img1, img2, threshold=0.5):
    """Matches features between two images using LoFTR."""
    # Convert numpy to Torch tensors [B, 1, H, W] in range [0, 1]
    t1 = torch.from_numpy(img1).float()[None, None] / 255.0
    t2 = torch.from_numpy(img2).float()[None, None] / 255.0
    
    with torch.no_grad():
        res = matcher({'image0': t1.to(device), 'image1': t2.to(device)})
        
    # Extract and filter by confidence threshold
    conf = res['confidence'].cpu().numpy()
    mask = conf > threshold
    
    return (
        res['keypoints0'].cpu().numpy()[mask],
        res['keypoints1'].cpu().numpy()[mask],
        conf[mask]
    )

print("⏳ Running Transformer-based matching...")
kpts_tmc, kpts_ohrc, confidences = extract_and_match(tmc_img, ohrc_img, threshold=0.4)
print(f"✅ Found {len(kpts_tmc)} robust matches across scale/illumination gaps.")

In [ ]:
def draw_correspondences(img1, img2, kp1, kp2, conf):
    """Draws side-by-side tie-lines color-coded by confidence."""
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    
    canvas = np.zeros((max(h1, h2), w1 + w2, 3), dtype=np.uint8)
    canvas[:h1, :w1] = cv2.cvtColor(img1.astype(np.uint8), cv2.COLOR_GRAY2BGR)
    canvas[:h2, w1:w1+w2] = cv2.cvtColor(img2.astype(np.uint8), cv2.COLOR_GRAY2BGR)
    
    # Normalize colormap for tie-lines
    norm_conf = (conf - conf.min()) / (conf.max() - conf.min() + 1e-5)
    cmap = plt.cm.plasma
    
    # Sort to draw highest confidence on top
    indices = np.argsort(conf)
    
    for i in indices:
        pt1 = (int(kp1[i][0]), int(kp1[i][1]))
        pt2 = (int(kp2[i][0] + w1), int(kp2[i][1]))
        color = tuple(int(c * 255) for c in cmap(norm_conf[i])[:3])
        
        cv2.line(canvas, pt1, pt2, color, 1, cv2.LINE_AA)
        cv2.circle(canvas, pt1, 3, color, -1)
        cv2.circle(canvas, pt2, 3, color, -1)
        
    return canvas

vis_img = draw_correspondences(tmc_img, ohrc_img, kpts_tmc, kpts_ohrc, confidences)

plt.figure(figsize=(18, 8))
plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
plt.title(f'Multi-Scale Correspondence via LoFTR (TMC → OHRC) | Matches: {len(confidences)}', 
          fontsize=16, fontweight='bold')
plt.axis('off')
plt.show()

---
## 🧊 4. Subsurface Ice Classification (Random Forest)

Lumina isn't just about imagery; it's a scientific analysis platform. We use **Dual-Frequency Synthetic Aperture Radar (DFSAR)** data (L-band and S-band) alongside thermal profiles to predict subsurface ice deposits in Permanently Shadowed Regions (PSRs) near the South Pole (Shiv Shakti Point).

- **CPR (Circular Polarization Ratio):** High CPR often indicates rough ice.
- **DOP (Degree of Polarization):** Used to distinguish ice from rocky ejecta.
- **Thermal Profiling:** Ice is unstable above ~110K.

In [ ]:
def generate_dfsar_telemetry(n_samples=5000):
    """Generates synthetic dataset mimicking DFSAR polarimetry for training."""
    np.random.seed(42)
    
    cpr = np.random.uniform(0.1, 1.2, n_samples)
    dop = np.random.uniform(0.2, 0.9, n_samples)
    temp_k = np.random.uniform(40, 250, n_samples)
    slope_deg = np.random.uniform(0, 30, n_samples)
    roughness = np.random.uniform(0.01, 0.15, n_samples)
    
    # Scientific Ground Truth Logic: High CPR + Low DOP + Extreme Cold = Ice
    ice_condition = (cpr > 0.75) & (dop < 0.55) & (temp_k < 120) & (slope_deg < 15)
    
    # Introduce noise (5% random error to simulate sensor noise)
    noise = np.random.random(n_samples) < 0.05
    labels = np.logical_xor(ice_condition, noise).astype(int)
    
    return pd.DataFrame({
        'cpr': cpr, 'dop': dop, 'temperature_k': temp_k, 
        'slope_deg': slope_deg, 'roughness': roughness, 'ice_present': labels
    })

df_science = generate_dfsar_telemetry()

# Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df_science.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('DFSAR & Thermal Feature Correlation', fontweight='bold')
plt.show()

In [ ]:
# Train Random Forest Classifier
X = df_science.drop(columns=['ice_present'])
y = df_science['ice_present']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)

preds = rf_model.predict(X_test)
print(f"🧊 Model Accuracy: {accuracy_score(y_test, preds):.2%}")
print("\nClassification Report:\n", classification_report(y_test, preds))

---
## 📡 5. Space Data Integration & Live Inference

Simulating real-time telemetry streaming from the Pragyan-2 Rover at Shiv Shakti Point. We feed live environmental readings into our ML model to update the Lumina Dashboard dynamically.

In [ ]:
def fetch_live_rover_telemetry():
    """Simulates live telemetry from Shiv Shakti Point (Crater Edge PSR)."""
    return {
        "timestamp_utc": datetime.utcnow().isoformat(),
        "location": {"lat": -69.373, "lon": 32.319, "name": "Shiv Shakti Point (PSR)"},
        "sensor_readings": {
            "cpr": 0.88,
            "dop": 0.42,
            "temperature_k": 95.0,  # Deep shadow, extremely cold
            "slope_deg": 8.5,
            "roughness": 0.04
        }
    }

telemetry = fetch_live_rover_telemetry()
readings_df = pd.DataFrame([telemetry['sensor_readings']])

# Infer Ice Probability
ice_prob = rf_model.predict_proba(readings_df)[0][1]

dashboard_payload = {
    "telemetry": telemetry,
    "ml_analysis": {
        "ice_probability": f"{ice_prob:.1%}",
        "confidence": "High" if ice_prob > 0.8 else "Medium",
        "model": "RandomForest_DFSAR_v1"
    }
}

print("\n🛰️ [LIVE TELEMETRY STREAM TO LUMINA DASHBOARD]")
print(json.dumps(dashboard_payload, indent=2))

---
## 🏁 6. Export Pipeline State

We export the feature matching coordinates into a JSON file. The Lumina frontend (React/Three.js) fetches this file to render 3D holographic tie-lines between the orbiter data and the lunar surface mesh in real-time.

In [ ]:
export_data = [{
    'pair': 'tmc__ohrc',
    'n_matches': int(len(kpts_tmc)),
    'mean_confidence': float(confidences.mean()),
    'pts_a': kpts_tmc.tolist(),
    'pts_b': kpts_ohrc.tolist(),
    'confidence': confidences.tolist()
}]

with open('matches.json', 'w') as f:
    json.dump(export_data, f, indent=2)

print("✅ Pipeline complete. 'matches.json' exported for Lumina WebGL rendering.")